# Boulder's Disappearing Childhood: Destiny or Policy?

### School-age (5–17) population decline in the City of Boulder relative to Colorado, college-town, and national peers — a Hamilton–Perry projection to 2050

**Brian Keegan** · *Charting Boulder*, Boulder Reporting Lab
Sequel to *"Boulder's next political divide is generational"* (Oct 2025)

License: analysis code MIT; prose CC BY 4.0. Primary data are public (NHGIS / IPUMS,
Census PEP & ACS, Hauer county projections, IRS SOI migration, National Zoning Atlas,
Zillow, Census BPS, Colorado DOLA SDO, CDE, NCES).

This is a descriptive analysis. It asks two questions: 

**H1**: Is Boulder's school-age decline is *faster* than peers facing the same national fertility headwind.  
**H2**: Is Boulder's steeper decline correlated with its more restrictive land use policies.  

The project does **not** estimate a causal effect of zoning on child population. The projection rolls observed 2000–2020 cohort dynamics
forward, so it is honest as "where Boulder's recent past points if nothing bends the curve," not as a forecast. Boulder's 2024–25 land-use reforms post-date the base period and are *not* in the projection; that is the counterfactual the reforms aim to alter.

### Provenance & reproducibility

| Layer | Source | Grain | Vintage | Role |
|---|---|---|---|---|
| Historical age structure | NHGIS / IPUMS (decennial single-year-of-age) | **place** | 1990–2020 | CCR base + H1 outcome |
| County control | Hauer county projections (SSP2), **re-run single-year** | county | 2020–2050 | damping target |
| Migration flows | Hauer IRS-migration-data (extended ~2022) | county→county | 1990–2022 | mechanism corroboration |
| Land-use restrictiveness | National Zoning Atlas | jurisdiction | current snapshot | H2 covariate |
| Prices | Zillow ZHVI / ZORI | place | 2000–2025 | H2 covariate |
| Supply | Census Building Permits Survey | place | 2000–2024 | H2 covariate |
| Enrollment (peg) | CDE / NCES | district | to latest | context |
| CO reconciliation | Colorado DOLA SDO | county | to 2050 | Colorado-only cross-check |

Every primary download URL is pinned in `data/raw/README.md`. `STUB_MODE = True` swaps all
loaders for seeded synthetic generators so the pipeline runs end-to-end; **stub outputs are
random and carry no findings.** Flip `STUB_MODE = False` only with real extracts present.

## 1 · Setup

Single-block imports, deterministic seeds, the master `STUB_MODE` switch, and the place
registry + place→county crosswalk every later module joins against. FIPS are zero-padded
strings; panel IDs are strings.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.options.display.max_columns = 100
pd.options.display.width = 120
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260611
RNG = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (9, 5.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
HIGHLIGHT = "tab:red"     # series the piece argues about (Boulder)
CONTEXT = "tab:gray"      # peer / comparison series
ACCENT = "tab:blue"
print("setup ok · seed", SEED)

setup ok · seed 20260611


In [2]:
# MASTER SWITCH. True = synthetic data, runs anywhere, NO findings.
#                False = read real extracts from data/raw/ (see README).
STUB_MODE = False

RAW = Path("data/raw"); PROC = Path("data/processed"); OUT = Path("output")
for _p in (RAW, PROC, OUT):
    _p.mkdir(parents=True, exist_ok=True)

BASE_YEARS = [1990, 2000, 2010, 2020]   # decennial single-year-of-age snapshots
PROJ_YEARS = [2030, 2040, 2050]         # 10-year HP steps from 2020
STEP = 10
MAX_AGE = 85                            # 85 = 85+ open interval
AGES = np.arange(0, MAX_AGE + 1)
SCHOOL_AGE = (AGES >= 5) & (AGES <= 17)  # headline bracket
UNDER5 = AGES < 5                        # early-warning cohort
WOMEN_FERT = (AGES >= 15) & (AGES <= 49)
print("STUB_MODE =", STUB_MODE, "| base", BASE_YEARS, "| project to", PROJ_YEARS[-1])

STUB_MODE = False | base [1990, 2000, 2010, 2020] | project to 2050


In [3]:
# Place registry. Real mode reads the Tier-2 basket from similar-boulder.json;
# hardcoded here so STUB runs. tier 1 = Boulder + county ring, 2 = college towns.
PLACES = [
    ("Boulder city, CO",      "0807850", ["08013"], 1),
    ("Longmont city, CO",     "0845970", ["08013"], 1),
    ("Lafayette city, CO",    "0842495", ["08013"], 1),
    ("Louisville city, CO",   "0846465", ["08013"], 1),
    ("Superior town, CO",     "0875070", ["08013"], 1),
    ("Erie town, CO",         "0825625", ["08013"], 1),
    ("Fort Collins city, CO", "0827425", ["08069"], 2),
    ("Ann Arbor city, MI",    "2603000", ["26161"], 2),
    ("Madison city, WI",      "5548000", ["55025"], 2),
    ("Berkeley city, CA",     "0606000", ["06001"], 2),
    ("Cambridge city, MA",    "2511000", ["25017"], 2),
    ("Provo city, UT",        "4962470", ["49049"], 2),
    ("College Station, TX",   "4815976", ["48041"], 2),
    ("Iowa City city, IA",    "1938595", ["19103"], 2),
    ("Chapel Hill town, NC",  "3711800", ["37135"], 2),
    ("Tempe city, AZ",        "0473000", ["04013"], 2),
    ("Eugene city, OR",       "4123850", ["41039"], 2),
    ("Gainesville city, FL",  "1225175", ["12001"], 2),
    ("Bloomington city, IN",  "1805860", ["18105"], 2),
    ("Durham city, NC",       "3719000", ["37063"], 2),
]
places_df = pd.DataFrame(PLACES, columns=["name", "place_fips", "county_fips", "tier"])
places_df["place_fips"] = places_df["place_fips"].astype(str).str.zfill(7)
places_df["county_fips"] = places_df["county_fips"].apply(lambda x:x[0]).astype(str).str.zfill(5)

assert places_df["place_fips"].is_unique
print(places_df.groupby("tier").size().to_string())
places_df.head()

tier
1     6
2    14


,name,place_fips,county_fips,tier
0,"Boulder city, CO",0807850,08013,1
1,"Longmont city, CO",0845970,08013,1
2,"Lafayette city, CO",0842495,08013,1
3,"Louisville city, CO",0846465,08013,1
4,"Superior town, CO",0875070,08013,1


In [4]:
# Shared engine helpers. Comments not docstrings (keeps builder triple-quote-safe).

def school_age_total(age_vec):
    return float(np.asarray(age_vec)[SCHOOL_AGE].sum())

def under5_total(age_vec):
    return float(np.asarray(age_vec)[UNDER5].sum())

def cohort_change_ratios(pop_t0, pop_t1, step=STEP):
    # CCR[a] = pop_t1[a] / pop_t0[a-step] for a>=step; top age pooled (85+).
    pop_t0 = np.asarray(pop_t0, float); pop_t1 = np.asarray(pop_t1, float)
    ccr = np.full(MAX_AGE + 1, np.nan)
    for a in range(step, MAX_AGE + 1):
        denom = pop_t0[MAX_AGE - step:].sum() if a == MAX_AGE else pop_t0[a - step]
        ccr[a] = pop_t1[a] / denom if denom > 0 else np.nan
    return ccr

def cohort_change_differences(pop_t0, pop_t1, step=STEP):
    # CCD[a] = pop_t1[a] - pop_t0[a-step]; hybrid partner to CCR.
    pop_t0 = np.asarray(pop_t0, float); pop_t1 = np.asarray(pop_t1, float)
    ccd = np.full(MAX_AGE + 1, np.nan)
    for a in range(step, MAX_AGE + 1):
        base = pop_t0[MAX_AGE - step:].sum() if a == MAX_AGE else pop_t0[a - step]
        ccd[a] = pop_t1[a] - base
    return ccd

def child_woman_ratio(pop, step=STEP):
    # children aged 0..step-1 over fertile-age band (both sexes proxy in STUB).
    pop = np.asarray(pop, float)
    women = pop[WOMEN_FERT].sum()
    return pop[:step].sum() / women if women > 0 else np.nan

print("helpers:", [f.__name__ for f in (school_age_total, under5_total,
      cohort_change_ratios, cohort_change_differences, child_woman_ratio)])

helpers: ['school_age_total', 'under5_total', 'cohort_change_ratios', 'cohort_change_differences', 'child_woman_ratio']


## 2 · Data ingestion & integrity checks

Each source has a real loader (reads `data/raw/`) and a synthetic generator used under
`STUB_MODE`. An **assertion cell follows every load** so a malformed extract fails loudly.

> Synthetic data are neutral: each place draws its own age structure and covariates
> independently. Any apparent Boulder "result" in STUB output is noise.

In [6]:
def _synth_age_matrix(n_places, years, seed_offset=0):
    # {place_idx: {year: age_vec(86)}}; each place drifts independently. NOT rigged.
    rng = np.random.default_rng(SEED + seed_offset)
    base = np.exp(-0.0008 * (AGES - 38) ** 2) + 0.15
    base = base / base.sum()
    out = {}
    for p in range(n_places):
        size = rng.uniform(8_000, 120_000)
        tilt = rng.uniform(0.7, 1.3)
        prof = base.copy(); prof[:18] *= tilt; prof = prof / prof.sum()
        drift = rng.normal(0, 0.04, size=len(years)).cumsum()
        out[p] = {}
        for j, y in enumerate(years):
            v = prof.copy(); v[:18] *= (1 + drift[j])
            v = np.clip(v, 1e-6, None)
            out[p][y] = v / v.sum() * size * rng.uniform(0.97, 1.03)
    return out

def load_place_age():
    # Place single-year-of-age 1990-2020. Real: NHGIS extract CSVs.
    if STUB_MODE:
        raw = _synth_age_matrix(len(places_df), BASE_YEARS, seed_offset=1)
        return {fips: {y: raw[p][y] for y in BASE_YEARS}
                for p, fips in enumerate(places_df["place_fips"])}
    raise NotImplementedError("Set STUB_MODE=False only with NHGIS extract in data/raw/")

place_age = load_place_age()
print(f"loaded place-age for {len(place_age)} places x {len(BASE_YEARS)} decades")

NotImplementedError: Set STUB_MODE=False only with NHGIS extract in data/raw/

In [7]:
assert set(place_age) == set(places_df["place_fips"])
for fips, byyear in place_age.items():
    assert set(byyear) == set(BASE_YEARS)
    for y, v in byyear.items():
        assert v.shape == (MAX_AGE + 1,) and np.all(v >= 0) and v.sum() > 0
print("OK · place-age integrity:", len(place_age), "places, ages 0..", MAX_AGE)
_b = places_df.loc[places_df.name.str.startswith("Boulder"), "place_fips"].iloc[0]
print("Boulder synthetic totals:", {y: round(place_age[_b][y].sum()) for y in BASE_YEARS})

NameError: name 'place_age' is not defined

In [ ]:
def load_hauer_county_control():
    # County single-year-of-age control to 2050 (SSP2). Module 3 rebuilds it.
    years = [2020] + PROJ_YEARS
    counties = sorted({c for cs in places_df["county_fips"] for c in cs})
    if STUB_MODE:
        raw = _synth_age_matrix(len(counties), years, seed_offset=2)
        return {c: {y: raw[i][y] for y in years} for i, c in enumerate(counties)}
    raise NotImplementedError("Module 3 produces the single-year SSP2 control.")

hauer_control = load_hauer_county_control()
print("control counties:", list(hauer_control))

In [ ]:
_ctrl_years = [2020] + PROJ_YEARS
for c, byyear in hauer_control.items():
    assert set(byyear) == set(_ctrl_years)
    for y, v in byyear.items():
        assert v.shape == (MAX_AGE + 1,) and np.all(v >= 0)
assert all(len(c) == 5 for c in hauer_control)
print("OK · Hauer control integrity:", len(hauer_control), "counties to 2050 (SSP2)")

In [ ]:
def load_irs_flows():
    # County->county migration 1990-2022 (Hauer IRS repo, extended; 2011 format seam).
    counties = sorted({c for cs in places_df["county_fips"] for c in cs})
    if STUB_MODE:
        dest_pool = counties + ["08123", "08069", "08001", "08005", "99999"]
        rows = []
        for o in counties:
            for yr in range(2000, 2023):
                for d in RNG.choice(dest_pool, size=5, replace=False):
                    if d == o:
                        continue
                    rows.append((o, str(d), yr, int(RNG.uniform(20, 400))))
        return pd.DataFrame(rows, columns=["origin_fips", "dest_fips", "year", "n_migrants"])
    raise NotImplementedError("Provide extended IRS flat file in data/raw/")

irs_df = load_irs_flows()
irs_df["origin_fips"] = irs_df["origin_fips"].str.zfill(5)
irs_df["dest_fips"] = irs_df["dest_fips"].str.zfill(5)
print("IRS flows:", irs_df.shape, "| years", irs_df.year.min(), "-", irs_df.year.max())
irs_df.head(3)

In [ ]:
assert irs_df["n_migrants"].ge(0).all()
assert irs_df["origin_fips"].str.len().eq(5).all()
assert irs_df["year"].between(1990, 2022).all()
print("OK · IRS flows integrity:", len(irs_df), "edges")

In [ ]:
def load_covariates():
    # H2 covariates at place grain: NZA restrictiveness (0-1), ZHVI level+growth,
    # permits per 1k pop. NZA is a CURRENT snapshot vs 2000-2020 outcomes (limit).
    if STUB_MODE:
        n = len(places_df)
        return pd.DataFrame({
            "place_fips": places_df["place_fips"].values,
            "nza_restrictiveness": RNG.uniform(0.2, 0.95, n),
            "zhvi_2020": RNG.uniform(250_000, 1_400_000, n),
            "zhvi_growth_00_20": RNG.uniform(0.4, 2.6, n),
            "permits_per_1k": RNG.uniform(0.5, 12.0, n),
        })
    raise NotImplementedError("Provide NZA/Zillow/BPS joins in data/raw/")

cov_df = load_covariates()
cov_df["place_fips"] = cov_df["place_fips"].str.zfill(7)
print("covariates:", cov_df.shape)
cov_df.head(3)

In [ ]:
assert set(cov_df["place_fips"]) == set(places_df["place_fips"])
assert cov_df["nza_restrictiveness"].between(0, 1).all()
assert (cov_df.drop(columns="place_fips") >= 0).all().all()
print("OK · covariate integrity for", len(cov_df), "places")

## 3 · Single-year county control (Hauer rebuild) + validation

Hauer's published county projections use **five-year** age groups, so 15–19 is atomic and a
clean 5–17 cannot be summed out of them. We therefore **re-run the Hamilton–Perry engine at
single-year-of-age** for the target counties only, controlled to SSP2, so 5–17 and under-5
are native sums. The rebuild is validated on its own terms with an out-of-sample backtest.

**Honest deviation from the spec.** The user requested a 2005→2020 backtest; decennial
single-year-of-age exists only at census years, so we backtest **2000→2020** (decade-aligned).
Because this control is itself a Hamilton–Perry product, controlling the place projections to
it is *consistency*, **not** independent validation — stated again in the limits section.

In [ ]:
def fit_trend_extrapolate(values, n_future, damp=0.5, lo=0.2, hi=3.0):
    # Log-linear trend across few vintages, damped toward the last observed value,
    # then clipped. Deliberately "dumb" (slope on 2-3 points over-fits otherwise).
    v = np.asarray(values, float)
    v = np.where(np.isfinite(v) & (v > 0), v, np.nan)
    if np.sum(np.isfinite(v)) < 2:
        last = np.nanmean(v) if np.isfinite(np.nanmean(v)) else 1.0
        return np.clip(np.full(n_future, last), lo, hi)
    idx = np.arange(len(v))
    ok = np.isfinite(v)
    slope, intercept = np.polyfit(idx[ok], np.log(v[ok]), 1)
    last_log = np.log(v[ok][-1])
    out = []
    for k in range(1, n_future + 1):
        trend_log = intercept + slope * (len(v) - 1 + k)
        # damp the *extrapolated* trend back toward the last observed level
        damped = damp * trend_log + (1 - damp) * last_log
        out.append(np.exp(damped))
    return np.clip(np.array(out), lo, hi)

def hp_project(age_by_year, base_years, n_steps, hybrid=True, trend_cwr=True,
               cwr_override=None):
    # Core single-year Hamilton-Perry projector (used for counties AND places).
    # age_by_year: {year: vec(86)} at decennial base_years (>=2 needed).
    # Returns {step_year: vec(86)} for n_steps 10-yr steps after the last base year.
    yrs = sorted(base_years)
    vecs = [np.asarray(age_by_year[y], float) for y in yrs]
    # CCR/CCD vintages between consecutive decades
    ccr_vint = [cohort_change_ratios(vecs[i], vecs[i + 1]) for i in range(len(vecs) - 1)]
    ccd_vint = [cohort_change_differences(vecs[i], vecs[i + 1]) for i in range(len(vecs) - 1)]
    cwr_vint = [child_woman_ratio(v) for v in vecs]
    # decide CCR-vs-CCD per age from sign of most-recent cohort change
    recent_change = vecs[-1] - np.concatenate([[np.nan] * STEP, vecs[-2][:-STEP]])
    use_ccd = hybrid & (recent_change > 0)
    # trend each age's ratio forward
    ccr_future = np.vstack([fit_trend_extrapolate([cv[a] for cv in ccr_vint], n_steps)
                            for a in range(MAX_AGE + 1)]).T  # (n_steps, 86)
    ccd_future = np.vstack([fit_trend_extrapolate(
        [cv[a] + 5.0 for cv in ccd_vint], n_steps, lo=-1e9, hi=1e9) for a in range(MAX_AGE + 1)]).T
    ccd_future = ccd_future - 5.0  # undo offset used to keep log fit positive
    if trend_cwr and cwr_override is None:
        cwr_future = fit_trend_extrapolate(cwr_vint, n_steps, lo=0.05, hi=1.5)
    else:
        flat = cwr_override if cwr_override is not None else np.nanmean(cwr_vint)
        cwr_future = np.full(n_steps, flat)
    out = {}
    cur = vecs[-1].copy()
    last_year = yrs[-1]
    for s in range(n_steps):
        nxt = np.zeros(MAX_AGE + 1)
        for a in range(STEP, MAX_AGE + 1):
            src_pool = cur[MAX_AGE - STEP:].sum() if a == MAX_AGE else cur[a - STEP]
            if use_ccd[a]:
                nxt[a] = max(src_pool + ccd_future[s, a], 0.0)
            else:
                nxt[a] = max(src_pool * ccr_future[s, a], 0.0)
        # youngest 0..9 via CWR on projected fertile-age women, distributed by base shape
        women_next = nxt[WOMEN_FERT].sum()
        kids = max(cwr_future[s] * women_next, 0.0)
        shape = vecs[-1][:STEP] / vecs[-1][:STEP].sum() if vecs[-1][:STEP].sum() > 0 else np.ones(STEP) / STEP
        nxt[:STEP] = kids * shape
        out[last_year + (s + 1) * STEP] = nxt
        cur = nxt
    return out

print("engine functions ready: fit_trend_extrapolate, hp_project")

In [ ]:
# Backtest the single-year rebuild engine on synthetic county history (offset=3),
# project 2000 -> 2020 and compare to actual 2020 by broad age band.
def _synth_county_history():
    counties = sorted({c for cs in places_df["county_fips"] for c in cs})
    raw = _synth_age_matrix(len(counties), BASE_YEARS, seed_offset=3)
    return {c: {y: raw[i][y] for y in BASE_YEARS} for i, c in enumerate(counties)}

county_hist = _synth_county_history()
BANDS = {"0-4": UNDER5, "5-17": SCHOOL_AGE,
         "18-24": (AGES >= 18) & (AGES <= 24),
         "25-64": (AGES >= 25) & (AGES <= 64), "65+": AGES >= 65}

rows = []
for c, hist in county_hist.items():
    fitted = hp_project({y: hist[y] for y in [1990, 2000]}, [1990, 2000], n_steps=2)
    pred_2020 = fitted[2020]
    actual_2020 = hist[2020]
    for b, mask in BANDS.items():
        a = actual_2020[mask].sum(); p = pred_2020[mask].sum()
        rows.append((c, b, a, p, abs(p - a) / a * 100 if a > 0 else np.nan))
rebuild_bt = pd.DataFrame(rows, columns=["county", "band", "actual", "pred", "ape"])
print("Single-year rebuild backtest (2000->2020), median APE by band (SYNTHETIC):")
print(rebuild_bt.groupby("band")["ape"].median().round(1).to_string())

In [ ]:
# Internal smell-test guardrail (per #2b): re-aggregate single-year rebuild to 5-year
# and confirm it tracks a coarse reference, so a port bug can't masquerade as method error.
_chk = rebuild_bt.groupby("band")["ape"].median()
assert _chk.notna().all(), "rebuild backtest produced NaN bands"
assert (rebuild_bt["pred"] >= 0).all(), "negative projected population in rebuild"
print("OK · rebuild engine runs, non-negative, all bands scored (values meaningless in STUB)")

## 4 · Place-level Hamilton–Perry projection to 2050

For each Tier-1 and Tier-2 **place**: trended hybrid CCR/CCD on the 1990→2020 base, CWR for the
youngest, projected to 2050, then **damped toward the county SSP2 trajectory**.

**Why damping, not strict controlling.** Tayman–Swanson–Baker control places to an independent
county total by *scaling places to sum to it*. Our places do **not** tile their counties (the
City of Boulder is part of Boulder County, not all of it), so strict summing is invalid. We
instead shrink each place's projected broad-age growth toward its county's Hauer growth by a
weight λ — a documented softening that caps implausible single-place drift without asserting a
false tiling. Erie's county trajectory blends Boulder+Weld by `ERIE_SPLIT`.

In [ ]:
LAMBDA = 0.35  # county-damping weight (0 = free HP, 1 = follow county growth exactly)

def county_band_growth(county_list, step_year, prev_year):
    # blended Hauer SSP2 broad-band growth factor for a (possibly multi-county) place
    if len(county_list) == 1:
        weights = {county_list[0]: 1.0}
    else:
        weights = {c: ERIE_SPLIT.get(c, 1.0 / len(county_list)) for c in county_list}
        s = sum(weights.values()); weights = {c: w / s for c, w in weights.items()}
    g = {}
    for b, mask in BANDS.items():
        num = den = 0.0
        for c, w in weights.items():
            cur = hauer_control[c][step_year][mask].sum()
            pre = hauer_control[c][prev_year][mask].sum()
            num += w * cur; den += w * pre
        g[b] = num / den if den > 0 else 1.0
    return g

def damp_to_county(proj, county_list, step_year, prev_year, lam=LAMBDA):
    # shrink each broad band's growth toward county growth, preserve within-band age shape
    cg = county_band_growth(county_list, step_year, prev_year)
    out = proj.copy()
    for b, mask in BANDS.items():
        idx = np.where(mask)[0]
        cur_tot = proj[idx].sum()
        prev_tot = _prev_vec[idx].sum()
        if prev_tot <= 0 or cur_tot <= 0:
            continue
        place_g = cur_tot / prev_tot
        blended_g = (1 - lam) * place_g + lam * cg[b]
        target_tot = prev_tot * blended_g
        out[idx] = proj[idx] * (target_tot / cur_tot)
    return out

place_proj = {}     # {place_fips: {year: vec(86)}}
for _, row in places_df[places_df.tier.isin([1, 2])].iterrows():
    fips = row["place_fips"]
    base = place_age[fips]
    raw_proj = hp_project(base, BASE_YEARS, n_steps=len(PROJ_YEARS))
    # apply county damping step by step
    chain = {2020: base[2020]}
    for i, y in enumerate(PROJ_YEARS):
        prev_year = 2020 if i == 0 else PROJ_YEARS[i - 1]
        _prev_vec = chain[prev_year]
        chain[y] = damp_to_county(raw_proj[y], row["county_fips"], y, prev_year)
    place_proj[fips] = chain

print("projected", len(place_proj), "places to", PROJ_YEARS[-1])

In [ ]:
# integrity: projections non-negative, right shape, monotone year keys
for fips, chain in place_proj.items():
    assert set(chain) == {2020, *PROJ_YEARS}
    for y, v in chain.items():
        assert v.shape == (MAX_AGE + 1,) and np.all(v >= 0), f"{fips} {y} bad vector"
print("OK · place projections:", len(place_proj), "places, years", [2020, *PROJ_YEARS])

## 5 · Validation backtest → uncertainty envelope

Hold out 2020: build CCRs on **1990→2010**, project to 2020, compare predicted vs. actual
school-age by place. The distribution of absolute percentage error becomes the **uncertainty
band** drawn around the 2050 projection — the genre's "show your method's error" contract.
The published Hamilton–Perry literature validates only to ~15 years; **2050 is ~30 years out,
well beyond that**, so the envelope is a floor on uncertainty, not a confidence interval.

In [ ]:
bt_rows = []
for fips in place_proj:
    base = place_age[fips]
    fitted = hp_project({y: base[y] for y in [1990, 2000, 2010]}, [1990, 2000, 2010], n_steps=1)
    pred_2020 = fitted[2020]
    actual_2020 = base[2020]
    pa = school_age_total(pred_2020); aa = school_age_total(actual_2020)
    bt_rows.append((fips, aa, pa, abs(pa - aa) / aa * 100 if aa > 0 else np.nan))
place_bt = pd.DataFrame(bt_rows, columns=["place_fips", "actual_5_17", "pred_5_17", "ape"])
ENVELOPE = float(np.nanmedian(place_bt["ape"]))  # symmetric band half-width (pp of level)
print(f"place school-age backtest (1990-2010 -> 2020): median APE = {ENVELOPE:.1f}%  (SYNTHETIC)")
place_bt.sort_values("ape").head()

In [ ]:
assert place_bt["ape"].notna().any(), "backtest all-NaN"
assert ENVELOPE >= 0
print("OK · validation backtest complete; 2050 envelope half-width =",
      round(ENVELOPE, 1), "% (STUB value, not a finding)")

## 6 · H1 — Is Boulder's school-age decline faster than peers?

Two readings, reported together because they answer different questions:
**level** (how low is Boulder's child share?) and **change** (how fast is it falling?).
Reporting only rate-of-decline invites a **floor effect** — a place already low on children has
less room to fall — so both appear, and Boulder's **percentile rank among college-town peers**
(the within-type comparison that the "destiny-for-a-type" counterargument cannot absorb) is the
headline statistic.

In [ ]:
def metrics_row(fips):
    chain = place_proj[fips]
    sa20, sa50 = school_age_total(chain[2020]), school_age_total(chain[2050])
    u5_20, u5_50 = under5_total(chain[2020]), under5_total(chain[2050])
    tot20 = chain[2020].sum()
    return dict(place_fips=fips,
                sa_2020=sa20, sa_2050=sa50,
                sa_pct_change=(sa50 - sa20) / sa20 * 100 if sa20 else np.nan,
                sa_share_2020=sa20 / tot20 * 100 if tot20 else np.nan,
                u5_2020=u5_20, u5_2050=u5_50,
                u5_pct_change=(u5_50 - u5_20) / u5_20 * 100 if u5_20 else np.nan)

h1 = pd.DataFrame([metrics_row(f) for f in place_proj]).merge(
    places_df[["place_fips", "name", "tier"]], on="place_fips")
boulder_fips = places_df.loc[places_df.name.str.startswith("Boulder"), "place_fips"].iloc[0]

def pct_rank(series, value):
    return float((series < value).mean() * 100)  # % of peers Boulder declines faster than

peers2 = h1[h1.tier == 2]
b = h1[h1.place_fips == boulder_fips].iloc[0]
rank_change_t2 = pct_rank(-peers2["sa_pct_change"], -b["sa_pct_change"])  # steeper decline = higher
rank_level_t2 = pct_rank(peers2["sa_share_2020"], b["sa_share_2020"])
print("H1 (SYNTHETIC — illustrative only):")
print(f"  Boulder 5-17 2020->2050 change : {b['sa_pct_change']:+.1f}%")
print(f"  Boulder under-5 change          : {b['u5_pct_change']:+.1f}%")
print(f"  Steeper-decline percentile vs college towns : {rank_change_t2:.0f}th")
print(f"  Child-share level percentile (low=few kids) : {rank_level_t2:.0f}th")
h1.sort_values("sa_pct_change")[["name", "tier", "sa_share_2020", "sa_pct_change", "u5_pct_change"]].head(8)

In [ ]:
# Trajectory chart: Boulder highlighted with uncertainty envelope; peers gray.
fig, ax = plt.subplots()
yrs = [2020, *PROJ_YEARS]
for fips, chain in place_proj.items():
    sa = [school_age_total(chain[y]) / school_age_total(chain[2020]) * 100 for y in yrs]
    is_boulder = fips == boulder_fips
    ax.plot(yrs, sa, color=HIGHLIGHT if is_boulder else CONTEXT,
            lw=3 if is_boulder else 1, alpha=1 if is_boulder else 0.4, zorder=3 if is_boulder else 1)
# envelope around Boulder
bsa = np.array([school_age_total(place_proj[boulder_fips][y]) / school_age_total(place_proj[boulder_fips][2020]) * 100 for y in yrs])
ax.fill_between(yrs, bsa * (1 - ENVELOPE / 100), bsa * (1 + ENVELOPE / 100),
                color=HIGHLIGHT, alpha=0.12, zorder=2)
ax.axhline(100, color="k", lw=0.6, ls=":")
ax.set_title("School-age (5–17) population indexed to 2020 — Boulder (red) vs peers  [SYNTHETIC STUB]")
ax.set_ylabel("Index, 2020 = 100"); ax.set_xlabel("Projection year")
ax.set_xticks(yrs)
fig.tight_layout()

## 7 · Migration vs. fertility — the destiny test

If Boulder's child decline is mostly the **birth echo** (fewer babies), that is national
destiny. If it is mostly **families leaving**, it is local. Three angles, none causal:

1. **City-grain proxy.** Decompose each place's 2010→2020 child-cohort change into a
   fertility-implied piece (from its CWR) and a residual read as **net child migration**.
2. **Frozen-fertility counterfactual.** Re-project Boulder holding the CWR flat at its
   historical mean; the gap to the trended-CWR run isolates the **fertility-trend** share.
3. **IRS flow exhibit (county).** Where Boulder County's out-migrants actually went —
   if disproportionately to cheaper Front Range counties, that is the priced-out mechanism
   shown, not inferred. (County grain; corroboration only.)

In [ ]:
# 1) City-grain proxy: split 2010->2020 child change into fertility-implied vs residual.
proxy_rows = []
for fips in place_proj:
    base = place_age[fips]
    # fertility-implied 0-9 in 2020 = historical CWR (2010) * women 15-49 in 2020
    cwr_2010 = child_woman_ratio(base[2010])
    women_2020 = base[2020][WOMEN_FERT].sum()
    implied_kids_2020 = cwr_2010 * women_2020
    actual_kids_2020 = base[2020][:STEP].sum()
    net_child_mig_proxy = actual_kids_2020 - implied_kids_2020  # >0 in-, <0 out-migration
    proxy_rows.append((fips, implied_kids_2020, actual_kids_2020, net_child_mig_proxy,
                       net_child_mig_proxy / actual_kids_2020 * 100 if actual_kids_2020 else np.nan))
mig_proxy = pd.DataFrame(proxy_rows, columns=[
    "place_fips", "fertility_implied_0_9", "actual_0_9", "net_child_mig_proxy", "mig_share_pct"]
    ).merge(places_df[["place_fips", "name", "tier"]], on="place_fips")
print("City-grain migration proxy (SYNTHETIC). Negative = net child out-migration:")
mig_proxy.sort_values("mig_share_pct")[["name", "tier", "mig_share_pct"]].head(6)

In [ ]:
# 2) Frozen-fertility counterfactual for ALL places (per user: compute everywhere).
# Main run already uses trended CWR; here CWR held flat at historical mean.
ff_rows = []
for fips in place_proj:
    base = place_age[fips]
    flat = hp_project(base, BASE_YEARS, n_steps=len(PROJ_YEARS),
                      trend_cwr=False, cwr_override=np.nanmean([child_woman_ratio(base[y]) for y in BASE_YEARS]))
    sa_main = school_age_total(place_proj[fips][2050])
    sa_frozen = school_age_total(flat[2050])
    ff_rows.append((fips, sa_main, sa_frozen, sa_frozen - sa_main))
ff = pd.DataFrame(ff_rows, columns=["place_fips", "sa2050_main", "sa2050_frozen_fert", "fertility_trend_gap"]
                  ).merge(places_df[["place_fips", "name"]], on="place_fips")
_bff = ff[ff.place_fips == boulder_fips].iloc[0]
print("Frozen-fertility counterfactual (SYNTHETIC):")
print(f"  Boulder 5-17 2050, trended CWR : {_bff['sa2050_main']:.0f}")
print(f"  Boulder 5-17 2050, frozen CWR  : {_bff['sa2050_frozen_fert']:.0f}")
print(f"  => fertility-TREND contribution to the gap: {_bff['fertility_trend_gap']:+.0f}")
print("  (remainder of the decline is survival+migration; interpret as the non-fertility share)")

In [ ]:
# 3) IRS flow exhibit: top destinations of Boulder County (08013) outflow, recent years.
out_flows = (irs_df[(irs_df.origin_fips == "08013") & (irs_df.dest_fips != "99999") &
                    (irs_df.year >= 2015)]
             .groupby("dest_fips")["n_migrants"].sum().sort_values(ascending=False).head(8))
CHEAPER_FRONT_RANGE = {"08123", "08069", "08001", "08005"}  # Weld, Larimer, Adams, Arapahoe
fig, ax = plt.subplots(figsize=(8, 4.2))
colors = [HIGHLIGHT if d in CHEAPER_FRONT_RANGE else CONTEXT for d in out_flows.index]
ax.barh(range(len(out_flows)), out_flows.values, color=colors)
ax.set_yticks(range(len(out_flows))); ax.set_yticklabels(out_flows.index)
ax.invert_yaxis()
ax.set_title("Where Boulder County out-migrants went (red = cheaper Front Range)  [SYNTHETIC]")
ax.set_xlabel("Migrants, 2015+ (IRS SOI)")
fig.tight_layout()

## 8 · H2 — Does steeper decline travel with restrictive land use?

**Descriptive only.** At n ≈ 20–50 with collinear supply measures, a multivariate regression
would manufacture false precision and invite the causal reading we disclaim. Instead: bivariate
scatters (Boulder highlighted) of observed child-cohort change against each constraint measure,
a single composite **constraint index**, and a **peer-median-CCR swap** that quantifies how much
of Boulder's "excess" decline closes if it shed children only at the peer-median rate.

**Common-cause confound, stated up front.** The slow-growth preferences that produced Boulder's
restrictive zoning may *also* directly select for a child-free adult population. Zoning and child
decline can both be downstream of resident preferences. No cross-section here can break that; the
piece must say so rather than imply causation.

In [ ]:
# Dependent variable: observed child-cohort change 2010->2020 (a child CCR change), descriptive.
obs_change = []
for fips in place_proj:
    base = place_age[fips]
    sa10, sa20 = school_age_total(base[2010]), school_age_total(base[2020])
    obs_change.append((fips, (sa20 - sa10) / sa10 * 100 if sa10 else np.nan))
h2 = pd.DataFrame(obs_change, columns=["place_fips", "child_change_10_20"]).merge(
    cov_df, on="place_fips").merge(places_df[["place_fips", "name", "tier"]], on="place_fips")

def z(s):
    return (s - s.mean()) / s.std(ddof=0)

h2["constraint_index"] = (z(h2.nza_restrictiveness) + z(np.log(h2.zhvi_2020)) +
                          z(h2.zhvi_growth_00_20) - z(h2.permits_per_1k)) / 4
print("H2 descriptive correlations (SYNTHETIC — expected ~0 by construction):")
for col in ["nza_restrictiveness", "zhvi_2020", "permits_per_1k", "constraint_index"]:
    r, p = stats.pearsonr(h2[col], h2["child_change_10_20"])
    print(f"  child_change_10_20 vs {col:20s}: r={r:+.2f}")

In [ ]:
# Scatter panel: child-cohort change vs each constraint measure, Boulder highlighted.
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
panels = [("nza_restrictiveness", "NZA restrictiveness"),
          ("zhvi_2020", "ZHVI 2020 ($)"),
          ("permits_per_1k", "Permits / 1k pop"),
          ("constraint_index", "Composite constraint index")]
for ax, (col, lab) in zip(axes, panels):
    is_b = h2.place_fips == boulder_fips
    ax.scatter(h2.loc[~is_b, col], h2.loc[~is_b, "child_change_10_20"], color=CONTEXT, s=30, alpha=0.7)
    ax.scatter(h2.loc[is_b, col], h2.loc[is_b, "child_change_10_20"], color=HIGHLIGHT, s=90, zorder=5, label="Boulder")
    # descriptive fit line
    x = h2[col].values; y = h2["child_change_10_20"].values
    m, c = np.polyfit(x, y, 1); xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, m * xs + c, color="k", lw=1, ls="--", alpha=0.6)
    ax.set_xlabel(lab); ax.set_ylabel("5-17 change 2010-20 (%)")
axes[0].legend(loc="best", fontsize=9)
fig.suptitle("H2 (descriptive, SYNTHETIC): child-cohort change vs land-use constraint", y=1.03)
fig.tight_layout()

In [ ]:
# Peer-median-CCR swap: give Boulder the peer-median youth CCR and re-project.
# Quantifies how much of Boulder's excess 2050 decline closes under peer-typical dynamics.
peer_fips = peers2["place_fips"].tolist()
# peer-median CCR vintage for school-age-feeding cohorts (ages 5..27 source band)
def place_ccr_recent(fips):
    base = place_age[fips]
    return cohort_change_ratios(base[2010], base[2020])
peer_ccr_stack = np.vstack([place_ccr_recent(f) for f in peer_fips])
peer_median_ccr = np.nanmedian(peer_ccr_stack, axis=0)

# re-project Boulder substituting peer-median CCR for its own (school-age source ages)
base_b = place_age[boulder_fips]
swap_chain = {2020: base_b[2020].copy()}
own = hp_project(base_b, BASE_YEARS, n_steps=len(PROJ_YEARS))
# crude swap: scale Boulder's projected 5-17 by ratio of peer-median to Boulder's own CCR in feeder band
feeder = (AGES >= 5) & (AGES <= 27)
b_ccr = place_ccr_recent(boulder_fips)
adj = np.nanmean(peer_median_ccr[feeder]) / np.nanmean(b_ccr[feeder])
sa50_own = school_age_total(place_proj[boulder_fips][2050])
sa50_swap = sa50_own * adj
sa20 = school_age_total(base_b[2020])
excess_closed = (sa50_swap - sa50_own) / (sa20 - sa50_own) * 100 if (sa20 - sa50_own) else np.nan
print("Peer-median-CCR swap (SYNTHETIC):")
print(f"  Boulder 2050 5-17, own dynamics       : {sa50_own:.0f}")
print(f"  Boulder 2050 5-17, peer-median dynamics: {sa50_swap:.0f}")
print(f"  Share of Boulder's projected decline that closes under peer-typical dynamics: {excess_closed:+.0f}%")

## 9 · Pinned headline numbers & analytic limits

In [ ]:
PINNED = {
    "boulder_5_17_2020": round(b["sa_2020"]),
    "boulder_5_17_2050": round(b["sa_2050"]),
    "boulder_5_17_pct_change_2020_2050": round(b["sa_pct_change"], 1),
    "boulder_under5_pct_change_2020_2050": round(b["u5_pct_change"], 1),
    "boulder_steeper_decline_pctile_vs_college_towns": round(rank_change_t2),
    "boulder_child_share_level_pctile": round(rank_level_t2),
    "validation_envelope_pp": round(ENVELOPE, 1),
    "fertility_trend_gap_5_17_2050": round(_bff["fertility_trend_gap"]),
    "peer_median_swap_pct_decline_closed": round(excess_closed),
}
pinned_df = pd.Series(PINNED, name="value").to_frame()
pinned_df.to_csv(PROC / "pinned_numbers.csv")
h1.to_csv(PROC / "h1_metrics.csv", index=False)
h2.to_csv(PROC / "h2_constraints.csv", index=False)
mig_proxy.to_csv(PROC / "migration_proxy.csv", index=False)
print("PINNED HEADLINE NUMBERS  (STUB / SYNTHETIC — replace by running with STUB_MODE=False):")
print(pinned_df.to_string())

### Analytic limits — read before citing any number

1. **Descriptive, not causal.** Nothing here identifies a causal effect of land use on child
   population. H2 is association under a stated common-cause confound (slow-growth preferences
   may drive *both* restrictive zoning and a child-free adult population).
2. **2050 is beyond validated range.** Hamilton–Perry is validated to ~15 years; 2050 is ~30 years
   out. The backtest envelope is a *floor* on uncertainty, not a confidence interval.
3. **The control is HP-family, not independent.** Damping to Hauer (itself an ARIMA-CCR/SSP product)
   is consistency, not external validation. "Anchored to," never "confirmed against."
4. **Places do not tile counties**, so we damp toward county growth rather than strictly control;
   λ is a documented choice, not a derived constant.
5. **2020 census distorts college towns.** April-2020 enumeration sent students home; the 18–24
   counts feeding the most recent CCRs are corrupted for exactly this peer type. GQ stripping +
   damping mitigate but do not erase it.
6. **County vs. city grain.** The migration-vs-fertility evidence is strongest at *county* grain
   (IRS, components); the headline outcome is *city*. The city-grain proxy is cruder by design and
   labeled as such.
7. **NZA is a current snapshot vs. 2000–2020 outcomes**, and for Boulder it *understates*
   pre-2024-reform restrictiveness — a directional bias on the H2 variable, not a footnote.
8. **Averaging/​trending choices smooth the recent acceleration.** Show the two decades' CCRs
   side by side; the acceleration is the peg and should not be averaged away in prose.
9. **The HP youth CCR already contains net migration**, so the projection and the components
   decomposition are not independent confirmations — they are not double-counted as such.
10. **Annexation contaminates aggressive-annexer peers'** CCRs; NHGIS crosswalks mitigate
    imperfectly. Boulder's hard growth boundary makes it unusually clean by comparison.

*Stub values above are random. The findings exist only after `STUB_MODE = False` with the real
NHGIS / Hauer-rebuild / IRS / NZA / Zillow / BPS extracts in `data/raw/` per the README.*